In [5]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import TwoLocal
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from scipy.optimize import minimize

from tqdm import tqdm
import pandas as pd

In [6]:
num_qubits = 12
num_layers = 4

def create_sk_hamiltonian(num_qubits):
    coefs = []
    zz_terms = []
    for i in range(num_qubits):
        for j in range(i+1, num_qubits):
            pauli_str = ['I'] * num_qubits
            pauli_str[i] = 'Z'
            pauli_str[j] = 'Z'
            
            coef = np.random.normal()/np.sqrt(num_qubits)
            
            coefs.append(coef)
            zz_terms.append(SparsePauliOp(''.join(pauli_str), coeffs=coef))
            
    
    # X terms (transverse field)
    x_terms = []
    for i in range(num_qubits):
        pauli_str = ['I']*num_qubits
        pauli_str[i] = 'X'
        x_terms.append(SparsePauliOp(''.join(pauli_str), coeffs=1))
    
    return coefs, sum(zz_terms + x_terms)

In [7]:
from qiskit.quantum_info import Statevector, Pauli, SparsePauliOp, StabilizerState, DensityMatrix

observables = []
for i in range(num_qubits):
    for j in range(i+1, num_qubits):
        pauli_str = ['I'] * num_qubits
        pauli_str[i] = 'Z'
        pauli_str[j] = 'Z'
        observables.append(Pauli(''.join(pauli_str)))
        
for i in range(num_qubits):
    pauli_str = ['I'] * num_qubits
    pauli_str[i] = 'X'
    observables.append(Pauli(''.join(pauli_str)))

In [21]:
ansatz = TwoLocal(
    num_qubits,
    rotation_blocks='ry',
    entanglement_blocks='cx',
    entanglement='circular',
    reps=num_layers,
    skip_final_rotation_layer=True,
    insert_barriers=True
)
df = pd.DataFrame({
    'coefs': [],
    'params': []
})

backend_noiseless = AerSimulator(method='statevector')

for _ in tqdm(range(100)):
    coefs, hamiltonian = create_sk_hamiltonian(num_qubits)

    initial_params = np.random.uniform(-np.pi, np.pi, ansatz.num_parameters)

    def noiseless_cost(params):
        bound_circuit = ansatz.assign_parameters(params)
    
        transpiled = transpile(bound_circuit, backend_noiseless)
        transpiled.save_statevector()
        job = backend_noiseless.run(transpiled)
        result = job.result()
        statevector = result.get_statevector()
    
        return np.real(statevector.expectation_value(hamiltonian))

    result_noiseless = minimize(
        noiseless_cost,
        initial_params,
        method='L-BFGS-B',
        options={'maxiter': 50}
    )

    optimal_params = result_noiseless.x

    new_row = pd.DataFrame({
    'coefs': [coefs],
    'params': [optimal_params]
    })

    df = pd.concat([df, new_row], ignore_index=True)

C:\Users\User\AppData\Local\Temp\ipykernel_42636\1126234216.py:1: DeprecationWarning: The class ``qiskit.circuit.library.n_local.two_local.TwoLocal`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.n_local instead.
  ansatz = TwoLocal(
100%|█████████████████████████████████████████████████████████████████████████████| 100/100 [8:45:06<00:00, 315.07s/it]


In [22]:
df.to_csv('./data/sk-hamiltonians.csv', index=False)

In [23]:
df

,coefs,params
0,"[0.08793102970898964, -0.23199292980645622, 0....","[-1.5762834744882814, -1.5711571789372893, -1...."
1,"[-0.17323903917032205, -0.023608927209224785, ...","[3.909114975346285, -3.1434732257851667, 2.937..."
2,"[-0.0036283157430498235, -0.07668870131061244,...","[-1.5436220752414793, 1.5989423605744757, -0.0..."
3,"[-0.3133326127692122, 0.28326643932744655, -0....","[-0.01351322008191654, 3.1418997846004566, 3.1..."
4,"[-0.06459802819387889, 0.22436593105057118, -0...","[1.744445907153414, -4.9102114883967145, 4.680..."
...,...,...
95,"[-0.3730468778672938, 0.2071951125811913, 0.39...","[-0.12244117567651548, 1.8185268224126707, -1...."
96,"[-0.05326325709330029, -0.07627266434778572, -...","[-3.1508615443913914, 3.1664160092283833, -2.2..."
97,"[0.3254693189390827, -0.5343220669159664, 0.18...","[-1.6016933515346923, -1.6120417830911493, -3...."
98,"[-0.12208655864280432, -0.2726249271937025, 0....","[-3.141136287443638, 3.141853192559041, 0.9973..."
